# 03 — Modelado
## LoanRisk-ML

Entrenamos y comparamos modelos de clasificación para predecir el default
en préstamos personales usando el dataset de Lending Club.

### Objetivos
- Realizar el split temporal train/val/test
- Aplicar ColumnTransformer con TargetEncoder correctamente después del split
- Comparar LightGBM vs XGBoost con tracking en MLflow
- Seleccionar el modelo con mejor balance AUC-PR y gap train-val

### Métrica principal
AUC-PR — más adecuada que AUC-ROC cuando hay desbalance de clases (80/20).

### Input
`data/processed/loan_features.parquet`

### Output
`mlruns/` — experimentos MLflow
`data/processed/X_train.parquet`, `X_val.parquet`, `X_test.parquet`
`data/processed/y_train.parquet`, `y_val.parquet`, `y_test.parquet`

## 1. Importaciones

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import mlflow
import mlflow.sklearn

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_recall_curve,
    roc_curve
)

from category_encoders import TargetEncoder
import lightgbm as lgb
import xgboost as xgb

print("Librerías cargadas correctamente")

Librerías cargadas correctamente


## 2. Rutas y carga del dataset

In [2]:
ROOT = Path('..').resolve()
DATA_PROCESSED = ROOT / 'data' / 'processed'
MODELS_DIR = ROOT / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(DATA_PROCESSED / 'loan_features.parquet')

print(f"Shape: {df.shape}")
print(f"Default rate: {df['target'].mean():.2%}")
print(f"\nTipos de datos:")
print(df.dtypes.value_counts())
print(f"\nColumnas string (pendientes de TargetEncoding):")
print(df.select_dtypes(include='object').columns.tolist())

Shape: (391164, 77)
Default rate: 20.15%

Tipos de datos:
float64    60
int64      10
object      4
int32       3
Name: count, dtype: int64

Columnas string (pendientes de TargetEncoding):
['home_ownership', 'verification_status', 'purpose', 'addr_state']


## 3. Split temporal train / val / test

Usamos validación temporal en lugar de split aleatorio.
Entrenamos en préstamos anteriores y validamos en préstamos posteriores
— simulando exactamente cómo funcionaría el modelo en producción.

In [4]:
# Separar features y target
X = df.drop(columns=['target'])
y = df['target']

# Split temporal: 70% train, 15% val, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print(f"Train: {X_train.shape} — default rate: {y_train.mean():.2%}")
print(f"Val:   {X_val.shape}   — default rate: {y_val.mean():.2%}")
print(f"Test:  {X_test.shape}  — default rate: {y_test.mean():.2%}")

Train: (273814, 76) — default rate: 20.15%
Val:   (58675, 76)   — default rate: 20.15%
Test:  (58675, 76)  — default rate: 20.15%


## 4. Construcción del ColumnTransformer

Definimos pipelines separados para columnas numéricas y categóricas.
El TargetEncoder se ajusta SOLO con datos de entrenamiento para evitar leakage.

In [5]:
# Identificar columnas por tipo
num_cols = X_train.select_dtypes(include='number').columns.tolist()
cat_cols = X_train.select_dtypes(include='object').columns.tolist()

print(f"Columnas numéricas: {len(num_cols)}")
print(f"Columnas categóricas: {len(cat_cols)}")
print(f"Categóricas: {cat_cols}")

# Pipeline numérico: imputar con mediana y escalar
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Pipeline categórico: TargetEncoder
categorical_transformer = Pipeline(steps=[
    ('target_encoder', TargetEncoder(cols=cat_cols))
])

# ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, num_cols),
    ('cat', categorical_transformer, cat_cols)
])

print("\nColumnTransformer definido correctamente")

Columnas numéricas: 72
Columnas categóricas: 4
Categóricas: ['home_ownership', 'verification_status', 'purpose', 'addr_state']

ColumnTransformer definido correctamente


## 5. Ajuste del preprocesador y transformación de datos

In [6]:
# Ajustar SOLO con X_train
# TargetEncoder aprende el default rate de cada categoría usando y_train
preprocessor.fit(X_train, y_train)

# Transformar los tres conjuntos
X_train_prep = preprocessor.transform(X_train)
X_val_prep   = preprocessor.transform(X_val)
X_test_prep  = preprocessor.transform(X_test)

print(f"X_train_prep: {X_train_prep.shape}")
print(f"X_val_prep:   {X_val_prep.shape}")
print(f"X_test_prep:  {X_test_prep.shape}")

X_train_prep: (273814, 76)
X_val_prep:   (58675, 76)
X_test_prep:  (58675, 76)


## 6. Configuración de MLflow

In [7]:
MLFLOW_DIR = ROOT / 'mlruns'

mlflow.set_tracking_uri(f"file:///{MLFLOW_DIR}")
mlflow.set_experiment("loanrisk-models")

print(f"Tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experimento: loanrisk-models")

2026/04/05 12:06:09 INFO mlflow.tracking.fluent: Experiment with name 'loanrisk-models' does not exist. Creating a new experiment.


Tracking URI: file:///C:\Users\micke\OneDrive\Desktop\projects\loanrisk-ml\mlruns
Experimento: loanrisk-models


In [8]:
models = {
    "LightGBM": lgb.LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=31,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    ),
    "XGBoost": xgb.XGBClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        random_state=42,
        n_jobs=-1,
        eval_metric='logloss',
        verbosity=0
    )
}

print("Modelos definidos:")
for name, model in models.items():
    print(f"  {name}")

Modelos definidos:
  LightGBM
  XGBoost


## 8. Loop de entrenamiento con MLflow tracking

In [9]:
import time

results = []

for name, model in models.items():
    print(f"\nEntrenando {name}...")

    with mlflow.start_run(run_name=name):

        # Entrenar y medir tiempo
        t0 = time.time()
        model.fit(X_train_prep, y_train)
        tiempo = time.time() - t0

        # Predicciones en probabilidad
        p_train = model.predict_proba(X_train_prep)[:, 1]
        p_val   = model.predict_proba(X_val_prep)[:, 1]

        # Métricas
        auc_pr_train = average_precision_score(y_train, p_train)
        auc_pr_val   = average_precision_score(y_val, p_val)
        auc_roc_train = roc_auc_score(y_train, p_train)
        auc_roc_val   = roc_auc_score(y_val, p_val)
        gap_train_val = auc_pr_train - auc_pr_val

        # KS Statistic
        fpr, tpr, _ = roc_curve(y_val, p_val)
        ks = (tpr - fpr).max()

        # Gini
        gini = 2 * auc_roc_val - 1

        # Loggear en MLflow
        mlflow.log_params({
            'model': name,
            'n_estimators': 500,
            'learning_rate': 0.05
        })
        mlflow.log_metrics({
            'auc_pr_train':  round(auc_pr_train, 4),
            'auc_pr_val':    round(auc_pr_val, 4),
            'auc_roc_train': round(auc_roc_train, 4),
            'auc_roc_val':   round(auc_roc_val, 4),
            'gap_train_val': round(gap_train_val, 4),
            'ks':            round(ks, 4),
            'gini':          round(gini, 4),
            'tiempo_seg':    round(tiempo, 2)
        })

        results.append({
            'Modelo':        name,
            'AUC-PR Train':  round(auc_pr_train, 4),
            'AUC-PR Val':    round(auc_pr_val, 4),
            'AUC-ROC Val':   round(auc_roc_val, 4),
            'KS':            round(ks, 4),
            'Gini':          round(gini, 4),
            'Gap Train-Val': round(gap_train_val, 4),
            'Tiempo (s)':    round(tiempo, 2)
        })

        print(f"  AUC-PR Train: {auc_pr_train:.4f}")
        print(f"  AUC-PR Val:   {auc_pr_val:.4f}")
        print(f"  AUC-ROC Val:  {auc_roc_val:.4f}")
        print(f"  KS:           {ks:.4f}")
        print(f"  Gini:         {gini:.4f}")
        print(f"  Gap:          {gap_train_val:.4f}")
        print(f"  Tiempo:       {tiempo:.1f}s")

print("\nEntrenamiento completado.")


Entrenando LightGBM...
  AUC-PR Train: 0.6310
  AUC-PR Val:   0.5623
  AUC-ROC Val:  0.7853
  KS:           0.4120
  Gini:         0.5706
  Gap:          0.0686
  Tiempo:       13.1s

Entrenando XGBoost...
  AUC-PR Train: 0.6435
  AUC-PR Val:   0.5628
  AUC-ROC Val:  0.7861
  KS:           0.4146
  Gini:         0.5723
  Gap:          0.0807
  Tiempo:       12.4s

Entrenamiento completado.


## 9. Tabla comparativa de resultados

In [10]:
df_results = pd.DataFrame(results)
df_results = df_results.sort_values('AUC-PR Val', ascending=False).reset_index(drop=True)

print("=" * 75)
print("RESULTADOS BASELINE")
print("=" * 75)
print(df_results.to_string(index=False))
print("=" * 75)
print(f"\nModelo con mejor AUC-PR Val: {df_results.iloc[0]['Modelo']}")
print(f"Modelo con menor Gap:        {df_results.sort_values('Gap Train-Val').iloc[0]['Modelo']}")

RESULTADOS BASELINE
  Modelo  AUC-PR Train  AUC-PR Val  AUC-ROC Val     KS   Gini  Gap Train-Val  Tiempo (s)
 XGBoost        0.6435      0.5628       0.7861 0.4146 0.5723         0.0807       12.41
LightGBM        0.6310      0.5623       0.7853 0.4120 0.5706         0.0686       13.07

Modelo con mejor AUC-PR Val: XGBoost
Modelo con menor Gap:        LightGBM


## 10. Guardar conjuntos procesados y preprocesador

In [11]:
import joblib

# Guardar conjuntos procesados
pd.DataFrame(X_train_prep).to_parquet(DATA_PROCESSED / 'X_train.parquet', index=False)
pd.DataFrame(X_val_prep).to_parquet(DATA_PROCESSED / 'X_val.parquet', index=False)
pd.DataFrame(X_test_prep).to_parquet(DATA_PROCESSED / 'X_test.parquet', index=False)

pd.Series(y_train.values).to_frame('target').to_parquet(DATA_PROCESSED / 'y_train.parquet', index=False)
pd.Series(y_val.values).to_frame('target').to_parquet(DATA_PROCESSED / 'y_val.parquet', index=False)
pd.Series(y_test.values).to_frame('target').to_parquet(DATA_PROCESSED / 'y_test.parquet', index=False)

# Guardar preprocesador
joblib.dump(preprocessor, MODELS_DIR / 'preprocessor.joblib')

print("Conjuntos guardados:")
print(f"  X_train: {X_train_prep.shape}")
print(f"  X_val:   {X_val_prep.shape}")
print(f"  X_test:  {X_test_prep.shape}")
print(f"\nPreprocesador guardado en models/preprocessor.joblib")

Conjuntos guardados:
  X_train: (273814, 76)
  X_val:   (58675, 76)
  X_test:  (58675, 76)

Preprocesador guardado en models/preprocessor.joblib
